## Imports and Configs

In [2]:
import pandas as pd
import numpy as np
from math import sqrt

ITER_CSV = r"../results/3_smoking_multi_iter_n500_50iters.csv"

# Fixed-by-construction values from the simulation code
N_TOTAL = 500
N_TRAIN = 400
N_TEST = 100
SIGMA = 3.0
TARGET_TREATMENT_PREV = 0.075

# Confidence level for the simulation-summary table
CI_LEVEL = 0.95

## Helpers

In [3]:
def t_critical_two_sided(conf_level: float, df: int) -> float:
    """
    Return two-sided t critical value.
    Falls back to normal approx if scipy is unavailable.
    """
    alpha = 1.0 - conf_level
    try:
        from scipy.stats import t
        return float(t.ppf(1 - alpha / 2, df))
    except Exception:
        return 1.645 if abs(conf_level - 0.90) < 1e-9 else 1.96

def mean_ci(series: pd.Series, conf_level: float = 0.90):
    """
    Mean and two-sided CI across iterations.
    """
    x = pd.to_numeric(series, errors="coerce").dropna().to_numpy()
    n = len(x)
    if n == 0:
        return np.nan, np.nan, np.nan, 0
    mean = float(np.mean(x))
    if n == 1:
        return mean, np.nan, np.nan, 1
    sd = float(np.std(x, ddof=1))
    se = sd / sqrt(n)
    tc = t_critical_two_sided(conf_level, n - 1)
    lo = mean - tc * se
    hi = mean + tc * se
    return mean, lo, hi, n

def fmt_num(x, digits=3):
    if pd.isna(x):
        return ""
    return f"{x:.{digits}f}"

def fmt_pct(x, digits=1):
    if pd.isna(x):
        return ""
    return f"{100*x:.{digits}f}\\%"

def fmt_ci(lo, hi, digits=3):
    if pd.isna(lo) or pd.isna(hi):
        return ""
    return f"[{lo:.{digits}f}, {hi:.{digits}f}]"

def fmt_ci_pct(lo, hi, digits=1):
    if pd.isna(lo) or pd.isna(hi):
        return ""
    return f"[{100*lo:.{digits}f}\\%, {100*hi:.{digits}f}\\%]"

def latex_escape(s: str) -> str:
    return (
        s.replace("\\", "\\textbackslash{}")
         .replace("%", "\\%")
         .replace("_", "\\_")
    )


df = pd.read_csv(ITER_CSV)

sim_cols = [
    "iteration",
    "seed",
    "treatment_rate",
    "n_treated_train",
    "n_treated_test",
    "sg1_n",
    "sg2_n",
    "sg3_n",
    "true_ate",
    "tau_sd",
    "true_sg1",
    "true_sg2",
    "true_sg3",
]

missing = [c for c in sim_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in CSV: {missing}")

# Keep one row per completed iteration
sim_df = df[sim_cols].drop_duplicates(subset=["iteration"]).copy()
sim_df = sim_df.sort_values("iteration").reset_index(drop=True)

n_valid = len(sim_df)

# Derived quantity
sim_df["snr_tau_over_sigma"] = sim_df["tau_sd"] / SIGMA

## Summary rows

In [4]:
rows = []

# fixed rows
rows.append({
    "Property": "Sample size, $n$",
    "Mean": f"{N_TOTAL}",
    "CI / Note": "fixed by construction"
})
rows.append({
    "Property": "Train / test split",
    "Mean": f"{N_TRAIN} / {N_TEST}",
    "CI / Note": "fixed by construction"
})
rows.append({
    "Property": "Noise level, $\\sigma$",
    "Mean": f"{SIGMA:.2f}",
    "CI / Note": "fixed by construction"
})
rows.append({
    "Property": "Target treatment prevalence",
    "Mean": f"{100*TARGET_TREATMENT_PREV:.1f}\\%",
    "CI / Note": "fixed by construction"
})

# estimated rows
summary_specs = [
    ("True ATE", "true_ate", "num3"),
    ("$\\tau^*(X)$ std deviation", "tau_sd", "num3"),
    ("\\% treated", "treatment_rate", "pct2"),
    ("Mean treated train-set size", "n_treated_train", "num1"),
    ("Mean treated test-set size", "n_treated_test", "num1"),
    ("SG1 true mean effect", "true_sg1", "num3"),
    ("SG2 true mean effect", "true_sg2", "num3"),
    ("SG3 true mean effect", "true_sg3", "num3"),
    ("Mean SG1 test-set size", "sg1_n", "num1"),
    ("Mean SG2 test-set size", "sg2_n", "num1"),
    ("Mean SG3 test-set size", "sg3_n", "num1"),
]

for label, col, kind in summary_specs:
    mean, lo, hi, n = mean_ci(sim_df[col], conf_level=CI_LEVEL)

    if kind == "num3":
        mean_str = fmt_num(mean, 3)
        ci_str = fmt_ci(lo, hi, 3)
    elif kind == "num1":
        mean_str = fmt_num(mean, 1)
        ci_str = fmt_ci(lo, hi, 1)
    elif kind == "pct2":
        mean_str = fmt_pct(mean, 2)
        ci_str = fmt_ci_pct(lo, hi, 2)
    else:
        raise ValueError(f"Unknown kind: {kind}")

    rows.append({
        "Property": label,
        "Mean": mean_str,
        "CI / Note": ci_str
    })

rows.append({
    "Property": "Valid iterations",
    "Mean": f"{n_valid} of 50",
    "CI / Note": f"{100*n_valid/50:.1f}\\% completed"
})

summary_table = pd.DataFrame(rows)

## Final

In [5]:
print(summary_table.to_string(index=False))

                   Property      Mean             CI / Note
           Sample size, $n$       500 fixed by construction
         Train / test split 400 / 100 fixed by construction
      Noise level, $\sigma$      3.00 fixed by construction
Target treatment prevalence     7.5\% fixed by construction
                   True ATE     3.230        [3.225, 3.236]
  $\tau^*(X)$ std deviation     2.248        [2.237, 2.258]
                 \% treated    5.62\%      [5.45\%, 5.79\%]
Mean treated train-set size      22.7          [21.8, 23.6]
 Mean treated test-set size       5.4            [4.7, 6.1]
       SG1 true mean effect     0.595        [0.570, 0.620]
       SG2 true mean effect     7.387        [7.286, 7.489]
       SG3 true mean effect     4.956        [4.727, 5.185]
     Mean SG1 test-set size      14.8          [13.7, 16.0]
     Mean SG2 test-set size       8.1            [7.5, 8.7]
     Mean SG3 test-set size      10.5           [9.8, 11.2]
           Valid iterations  44 of 50   